# AAIGM — Cleaned Research Notebook

This notebook demonstrates safe, reproducible language-model evaluation using the modules in `src/`. Harmful prompts and operational jailbreak targets from the original exploratory notebook have been removed.

In [ ]:
from src.model_loader import load_model, environment_info
from src.generation import generate_text
print(environment_info())
bundle = load_model('Qwen/Qwen3-1.7B')

In [ ]:
prompt = 'Explain the importance of reproducibility in machine-learning experiments.'
answer = generate_text(bundle.model, bundle.tokenizer, prompt, max_new_tokens=150)
print(answer)

## Token-level likelihood comparison

In [ ]:
from src.logprob import rank_options

question = 'What is the capital of France?'
options = ['Berlin', 'Madrid', 'Paris', 'Rome']
rank_options(bundle.model, bundle.tokenizer, question, options)

## Adversarial suffix research

The public code contains only the differentiable loss/candidate-token mechanics. Use synthetic benign targets for controlled experiments.

In [ ]:
from src.model_loader import load_model

bundle = load_model("Qwen/Qwen3-1.7B")

model = bundle.model
tokenizer = bundle.tokenizer

embedding_layer = model.get_input_embeddings()

print("Model loaded.")



# ------------------------------------------------
# Research prompt
# ------------------------------------------------

prompt = """
Explain why reproducibility is important in
machine-learning experiments.
"""

# ------------------------------------------------
# Target response used for the educational
# optimization experiment
# ------------------------------------------------

target = """
Reproducibility allows researchers to verify,
compare, and build upon experimental results.
"""


prompt_ids = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=True,
).input_ids.to(model.device)

target_ids = tokenizer(
    target,
    return_tensors="pt",
    add_special_tokens=False,
).input_ids.to(model.device)

print("Prompt tokens:", prompt_ids.shape[1])
print("Target tokens:", target_ids.shape[1])

In [ ]:
# Number of tokens that will be optimized
suffix_length = 8

# Start with a neutral/benign suffix
initial_suffix = " research context"

suffix_ids = tokenizer(
    initial_suffix,
    return_tensors="pt",
    add_special_tokens=False,
).input_ids.to(model.device)

# Make sure the suffix has the desired length
if suffix_ids.shape[1] != suffix_length:
    suffix_ids = torch.randint(
        low=0,
        high=tokenizer.vocab_size,
        size=(1, suffix_length),
        device=model.device,
    )

print(
    "Initial suffix:",
    repr(tokenizer.decode(suffix_ids[0]))
)

prompt_embeds = embedding_layer(prompt_ids)

target_embeds = embedding_layer(target_ids)

print("Prompt embeddings:", prompt_embeds.shape)
print("Target embeddings:", target_embeds.shape)

In [ ]:



from src.experiments.adversarial_suffix import (
    freeze_model,
    target_loss,
    candidate_token_ids,
)

import torch

num_optimization_steps = 120
top_k = 40

loss_history = []
suffix_history = []

print("Starting educational GCG optimization")
print("Initial suffix:", tokenizer.decode(suffix_ids[0]))

# ------------------------------------------------
# 1. Freeze the model
# ------------------------------------------------

freeze_model(model)


# ------------------------------------------------
# 2. Optimization loop
# ------------------------------------------------

for step in range(num_optimization_steps):

    # --------------------------------------------
    # A. Create differentiable suffix embeddings
    # --------------------------------------------

    suffix_embeds = (
        embedding_layer(suffix_ids)
        .detach()
        .clone()
        .requires_grad_(True)
    )

    # --------------------------------------------
    # B. Forward pass
    # --------------------------------------------

    full_embeds = torch.cat(
        [
            prompt_embeds,
            suffix_embeds,
            target_embeds.detach(),
        ],
        dim=1,
    )

    outputs = model(
        inputs_embeds=full_embeds,
        use_cache=False,
    )

    # --------------------------------------------
    # C. Calculate target loss
    # --------------------------------------------

    prefix_len = (
        prompt_embeds.shape[1]
        + suffix_ids.shape[1]
    )

    target_len = target_ids.shape[1]

    target_logits = outputs.logits[
        :,
        prefix_len - 1:
        prefix_len - 1 + target_len,
        :,
    ]

    loss = torch.nn.functional.cross_entropy(
        target_logits.reshape(-1, target_logits.size(-1)),
        target_ids.reshape(-1),
    )

    # --------------------------------------------
    # D. Calculate gradient
    # --------------------------------------------

    loss.backward()

    suffix_grad = suffix_embeds.grad.detach()

    current_loss = loss.item()

    # --------------------------------------------
    # E. Save experiment state
    # --------------------------------------------

    loss_history.append(current_loss)

    current_suffix = tokenizer.decode(
        suffix_ids[0]
    )

    suffix_history.append(current_suffix)

    print(
        f"Step {step:03d} | "
        f"Loss = {current_loss:.4f} | "
        f"Suffix = {current_suffix!r}"
    )

    # --------------------------------------------
    # F. Select suffix position
    # --------------------------------------------

    position = step % suffix_ids.shape[1]

    candidate_ids = candidate_token_ids(
        suffix_grad=suffix_grad,
        embedding_layer=embedding_layer,
        position=position,
        top_k=top_k,
    )

    # --------------------------------------------
    # G. Evaluate candidate tokens
    # --------------------------------------------

    best_loss = current_loss
    best_token = suffix_ids[0, position].item()

    for candidate in candidate_ids:

        candidate_suffix = suffix_ids.clone()

        candidate_suffix[0, position] = candidate

        candidate_loss = target_loss(
            model=model,
            prompt_embeds=prompt_embeds,
            suffix_ids=candidate_suffix,
            target_ids=target_ids,
            embedding_layer=embedding_layer,
        )

        candidate_loss_value = candidate_loss.item()

        if candidate_loss_value < best_loss:

            best_loss = candidate_loss_value
            best_token = candidate.item()

    # --------------------------------------------
    # H. Accept the best candidate
    # --------------------------------------------

    suffix_ids[0, position] = best_token

    # --------------------------------------------
    # I. Report update
    # --------------------------------------------

    if best_loss >= current_loss:

        print(
            f"  No improvement at position {position}"
        )

    else:

        print(
            f"  Updated position {position}: "
            f"{tokenizer.decode([best_token])!r} "
            f"| new loss = {best_loss:.4f}"
        )


# ------------------------------------------------
# 3. Final result
# ------------------------------------------------

print("\nOptimization complete.")

print(
    "Final suffix:",
    repr(tokenizer.decode(suffix_ids[0])),
)